# Projeto De Machine Learning - 2026.1

## Notebook 06 — Análise de Sentimento

### Projeto Kyra Pesquisa

*Maria Beatriz Ribeiro, Juliane Oliveira e Emanuel Gandra*

## 1. Objetivo

Este notebook implementa a **análise de sentimento** sobre os chunks preprocessados.

---

### Abordagem principal — Léxica (sem Deep Learning)

A análise de sentimento é realizada por meio de um **léxico de termos positivos e negativos**
adaptado ao contexto de entrevistas qualitativas: percepção de marca, experiência do consumidor,
recomendação e avaliação de produto.

**Por que léxica?**
- Sem uso de Deep Learning — interpretável e auditável
- A classificação vem de termos explícitos: é possível saber *exatamente por que* um chunk foi classificado como POS ou NEG
- Trata negações explícitas ("não gostei", "não recomendo", "não compraria") para evitar erros de classificação
- Funciona bem para o vocabulário de entrevistas qualitativas, sem necessidade de GPU nem de download de modelos pesados

Retorna para cada chunk:
- **sentimento_lexico**: `POS`, `NEG` ou `NEU`
- **score_lexico**: n_termos_pos − n_termos_neg (diretamente interpretável)
- **confianca_lexico**: intensidade relativa do score sobre o total de termos detectados

---

### Abordagem extra — BERT/pysentimiento (experimento adicional, não é a abordagem principal)

> ⚠️ **Atenção:** O pysentimiento usa um modelo Transformer (Deep Learning).
> Por isso, **não é a abordagem principal do projeto** — serve apenas como comparação exploratória.

O pysentimiento é mantido em uma seção separada, controlado pela variável `RODAR_BERT`.
Por padrão, `RODAR_BERT = False` e essa seção é ignorada completamente.

---

**Input:** `data/processed/interviews_chunks_modeling.parquet`
+ outputs do notebook 03 (`chunks_with_clusters_topics.parquet`)

**Output:** `outputs/sentimento/` com tabelas e gráficos prontos para o notebook 07

## 2. Checklist do Notebook

| Seção | O que entrega |
|---|---|
| **3. Setup** | Carregamento dos dados e configuração (`RODAR_BERT`) |
| **4. Abordagem Léxica** | Léxico, função de classificação e aplicação — **abordagem principal** |
| **5. Por projeto** | % POS/NEG/NEU por projeto — usando `sentimento_lexico` |
| **6. Por cluster** | Sentimento léxico cruzado com os clusters KMeans |
| **7. Por tópico NMF** | Sentimento léxico cruzado com os 18 tópicos temáticos |
| **8. Exemplos reais** | Trechos mais positivos e negativos por projeto (léxico) |
| **9. Experimento BERT** | pysentimiento (apenas se `RODAR_BERT = True`) — **não é a abordagem principal** |
| **10. Export** | Parquet + CSVs prontos para o notebook 07 |
| **11. Resumo** | Sumário consolidado dos resultados |

---
## 3. Setup e Carregamento

In [2]:
# ── Variável de controle ─────────────────────────────────────────────────
# RODAR_BERT = False  → executa apenas a abordagem léxica (padrão e recomendado)
# RODAR_BERT = True   → executa também o pysentimiento (Deep Learning, lento, ~3-4h em CPU)
RODAR_BERT = True

print(f'RODAR_BERT = {RODAR_BERT}')
if RODAR_BERT:
    print('  → A seção de experimento com pysentimiento (BERT) será executada na seção 9.')
    print('  → Atenção: pode demorar 3-4 horas em CPU.')
else:
    print('  → Apenas a abordagem léxica será executada (abordagem principal do projeto).')
    print('  → Para comparar com BERT, defina RODAR_BERT = True e re-execute.')

RODAR_BERT = True
  → A seção de experimento com pysentimiento (BERT) será executada na seção 9.
  → Atenção: pode demorar 3-4 horas em CPU.


In [3]:
import os
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

# ── Caminhos ────────────────────────────────────────────────────────────
_cwd = Path(os.path.abspath(''))
_root = _cwd
for _ in range(4):
    if (_root / 'data' / 'processed').exists():
        break
    _root = _root.parent

PROCESSED_DIR  = _root / 'data' / 'processed'
CLUSTER_BASE   = _root / 'outputs' / 'clusterizacao_insights_v2'
CLUSTER_RUN    = sorted(CLUSTER_BASE.glob('*'), reverse=True)[0]
SENT_DIR       = _root / 'outputs' / 'sentimento'
SENT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raiz do projeto : {_root}')
print(f'Run clusterização: {CLUSTER_RUN.name}')
print(f'Output sentimento: {SENT_DIR}')

Raiz do projeto : c:\Users\daiki\Desktop\kyrapesquisa-main\kyrapesquisa-main\Kyra Atual
Run clusterização: 20260520_113457
Output sentimento: c:\Users\daiki\Desktop\kyrapesquisa-main\kyrapesquisa-main\Kyra Atual\outputs\sentimento


In [4]:
# ── Carrega base principal ───────────────────────────────────────────────
df = pd.read_parquet(PROCESSED_DIR / 'interviews_chunks_modeling.parquet')
print(f'Base principal : {df.shape[0]:,} chunks | {df["projeto"].nunique()} projetos')

# ── Carrega chunks com clusters e tópicos NMF ────────────────────────────
df_clusters = pd.read_parquet(CLUSTER_RUN / 'chunks_with_clusters_topics.parquet')
print(f'Com clusters   : {df_clusters.shape[0]:,} chunks')
print(f'Colunas disponíveis: {list(df_clusters.columns)}')

# ── Coluna de texto para análise de sentimento ───────────────────────────
# Usamos text_for_embedding (texto mais próximo do original, sem stemming)
# Se não existir, fallback para text_for_keywords
TEXT_COL = 'text_for_embedding' if 'text_for_embedding' in df.columns else 'text_for_keywords'
print(f'Coluna de texto usada: {TEXT_COL}')
print(df[[TEXT_COL, 'projeto']].head(3))

Base principal : 4,976 chunks | 12 projetos
Com clusters   : 4,976 chunks
Colunas disponíveis: ['chunk_id', 'doc_id', 'cluster_label', 'cluster_auto_label', 'nmf_topic_id', 'nmf_auto_label', 'nmf_topic_weight', 'lda_topic_id', 'lda_auto_label', 'lda_topic_weight', 'embedding_x', 'embedding_y', 'projeto', 'publico', 'marca_foco', 'tipo_sessao', 'text_for_embedding', 'text_for_keywords', 'chunk_index', 'keep_for_modeling', 'quality_status', 'quality_reasons_str', 'idioma_detectado', 'score_pt', 'score_es', 'score_en', 'tokens_topic', 'n_words', 'n_topic_tokens', 'n_chars', 'lexical_diversity_topic', 'turn_index_start', 'turn_index_end', 'speaker_ids', 'n_speakers', 'question_unit_share', 'interviewer_word_share_heuristic', 'timestamp_start_seconds', 'timestamp_end_seconds', 'procedural_matches', 'procedural_score', 'source_filename_hash', 'collection_batch', 'mes', 'mes_num', 'ano', 'pais', 'canal_origem', 'contexto_origem', 'idioma_original', 'data_bruta_nome_arquivo', 'hora_bruta_nome_

---
## 4. Abordagem Léxica — Classificação de Sentimento (Abordagem Principal)

> **Esta é a abordagem oficial do projeto.**
> Classificação baseada em léxico de termos positivos e negativos adaptado ao contexto de
> entrevistas qualitativas. **Não usa Deep Learning** — é interpretável, auditável e roda sem GPU.

### Como funciona

1. Cada chunk é varrido em busca de termos do léxico positivo e do léxico negativo
2. **Negações** (`não`, `nunca`, `jamais`, `nem`, `nenhum`) detectadas até 3 palavras antes de um termo invertem sua polaridade — ex.: "não gostei" conta como negativo, não positivo
3. Expressões multi-palavra são verificadas primeiro (mais específicas), depois palavras individuais
4. **Score léxico** = n_termos_pos − n_termos_neg
5. **Classificação final**: `POS` se score > 0 | `NEG` se score < 0 | `NEU` se score = 0
6. **Confiança léxica**: |score| ÷ total de termos detectados (0 a 1)

### Por que não usar somente o léxico do notebook 04?

O notebook 04 usou um léxico manual de 62 palavras e obteve mediana = 0 (maioria NEU).
Aqui, o léxico foi expandido e adaptado ao contexto de pesquisa qualitativa de mercado,
incluindo expressões de recomendação, satisfação, crítica e negação.

In [5]:
# ── Léxico de Sentimento ─────────────────────────────────────────────────
# Adaptado ao contexto de entrevistas qualitativas:
# percepção de marca, experiência do consumidor, recomendação, avaliação de produto

TERMOS_POS = [
    # Avaliação direta positiva
    'ótimo', 'ótima', 'ótimos', 'ótimas',
    'excelente', 'excelentes', 'excepcional', 'excepcionais',
    'incrível', 'incríveis', 'maravilhoso', 'maravilhosa', 'maravilhosos', 'maravilhosas',
    'fantástico', 'fantástica', 'fantásticos', 'fantásticas',
    'perfeito', 'perfeita', 'perfeitos', 'perfeitas',
    'top', 'show', 'demais', 'legal', 'bacana',
    'bom', 'boa', 'bons', 'boas',
    'positivo', 'positiva', 'positivos', 'positivas',
    'melhor', 'melhores',
    # Afeto e preferência
    'adoro', 'adorei', 'amo', 'amei', 'gosto', 'gostei', 'gosta', 'gostamos',
    'adotei', 'adotaria', 'prefiro', 'preferi', 'preferiria',
    'curto', 'curti', 'curtia', 'curtiu',
    'aprecio', 'apreciei',
    # Recomendação
    'recomendo', 'recomendaria', 'recomendei', 'recomenda',
    'indicaria', 'indico', 'indiquei', 'indica',
    'vale a pena', 'vale muito', 'compro sempre', 'compraria novamente',
    'voltaria a comprar', 'compraria de novo',
    # Satisfação e bem-estar
    'satisfeito', 'satisfeita', 'satisfação', 'contente', 'feliz',
    'animado', 'animada', 'empolgado', 'empolgada',
    'me sinto bem', 'fico feliz', 'faz diferença', 'faz sentido',
    'me ajudou', 'me ajuda', 'resolveu meu', 'resolveu minha',
    # Qualidade e eficiência
    'qualidade', 'eficiente', 'eficaz', 'funciona', 'funcionou', 'resolve', 'resolveu',
    'prático', 'prática', 'práticos', 'práticas',
    'fácil', 'simples', 'conveniente', 'acessível',
    'rápido', 'rápida', 'agilidade', 'eficiência',
    'vale', 'valeu',
    # Experiência positiva
    'confiança', 'confiável', 'seguro', 'segura', 'segurança',
    'agradável', 'bonito', 'bonita', 'lindo', 'linda', 'elegante',
    'diferencial', 'inovador', 'inovadora', 'único', 'única',
    'superou', 'surpreendeu', 'surpreendente', 'surpreendeu positivamente',
    'benefício', 'vantagem', 'pontos positivos',
    'boa experiência', 'ótima experiência', 'experiência positiva',
    'bom atendimento', 'ótimo atendimento', 'atendimento excelente',
    'gostoso', 'delicioso', 'saboroso', 'cheiro bom', 'cheiro agradável',
    'recomendação', 'aprovado', 'aprovada',
]

TERMOS_NEG = [
    # Avaliação direta negativa
    'péssimo', 'péssima', 'péssimos', 'péssimas',
    'horrível', 'horríveis', 'terrível', 'terríveis',
    'ruim', 'ruins', 'fraco', 'fraca', 'fracos', 'fracas',
    'negativo', 'negativa', 'negativos', 'negativas',
    'decepcionante', 'decepcionantes', 'frustrante', 'frustrantes',
    'insatisfatório', 'insatisfatória',
    'pior', 'piores',
    # Emoções negativas
    'detesto', 'detestei', 'odeio', 'odiei', 'detestava', 'odiava',
    'insatisfeito', 'insatisfeita', 'insatisfação',
    'frustrado', 'frustrada', 'frustração',
    'decepcionado', 'decepcionada', 'decepção', 'decepcionei',
    'arrependimento', 'me arrependi', 'me arrependo', 'arrependida', 'arrependido',
    'desapontado', 'desapontada', 'desapontamento', 'desapontei',
    # Problemas e falhas
    'problema', 'problemas', 'defeito', 'defeitos', 'falha', 'falhas',
    'erro', 'erros', 'travou', 'quebrou', 'estragou', 'danificou', 'danifica',
    'não funciona', 'não resolve', 'não ajuda', 'não serve',
    # Reclamação e crítica
    'reclamar', 'reclamei', 'reclamação', 'reclamações', 'reclamo',
    'critico', 'critiquei', 'crítica negativa',
    'abusivo', 'abusiva', 'exploração', 'exploratório',
    'vergonhoso', 'vergonhosa', 'absurdo', 'absurda',
    'inaceitável', 'inadmissível', 'descaso',
    'enganoso', 'enganosa', 'propaganda enganosa', 'me enganaram',
    # Não recomendação
    'não recomendo', 'não recomendaria', 'não indicaria',
    'não compraria', 'não voltaria', 'não usaria', 'não voltaria a comprar',
    # Dificuldade e complexidade
    'difícil', 'difíceis', 'complicado', 'complicada', 'confuso', 'confusa',
    'demorado', 'demorada', 'demora', 'lento', 'lenta', 'atraso', 'demora',
    # Experiência negativa
    'ressecou', 'irritou', 'machucou', 'piorou', 'piorando',
    'mal atendido', 'mal atendida', 'descuidado', 'descuidada',
    'experiência ruim', 'experiência péssima', 'experiência horrível',
    'péssimo atendimento', 'atendimento ruim', 'atendimento péssimo',
    'cheiro ruim', 'cheiro forte demais', 'cheiro horrível',
    'fiquei com raiva', 'fui enganado', 'fui enganada',
]

NEGACOES = ['não', 'nao', 'nunca', 'jamais', 'nem', 'nenhum', 'nenhuma', 'tampouco']

print(f'Léxico carregado:')
print(f'  {len(TERMOS_POS)} termos positivos')
print(f'  {len(TERMOS_NEG)} termos negativos')
print(f'  {len(NEGACOES)} palavras de negação: {NEGACOES}')

Léxico carregado:
  145 termos positivos
  132 termos negativos
  8 palavras de negação: ['não', 'nao', 'nunca', 'jamais', 'nem', 'nenhum', 'nenhuma', 'tampouco']


In [6]:
import re

# ── Função de Classificação Léxica ──────────────────────────────────────
def _normaliza(texto):
    t = texto.lower()
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def classificar_lexico(texto, termos_pos=None, termos_neg=None,
                       negacoes=None, janela_negacao=3):
    """
    Classifica sentimento por abordagem léxica com tratamento de negações.
    Retorna: (sentimento, score_lexico, n_pos, n_neg, confianca_lexico)
    """
    if termos_pos is None:
        termos_pos = TERMOS_POS
    if termos_neg is None:
        termos_neg = TERMOS_NEG
    if negacoes is None:
        negacoes = NEGACOES

    texto_norm = _normaliza(texto)
    texto_trabalho = texto_norm  # será modificado ao remover multi-word matches
    n_pos = 0
    n_neg = 0

    # Primeiro: expressões multi-palavra (mais específicas — evita double-count)
    todos_termos = [(t, +1) for t in termos_pos if ' ' in t] + \
                   [(t, -1) for t in termos_neg if ' ' in t]
    todos_termos.sort(key=lambda x: len(x[0]), reverse=True)

    for termo, polaridade in todos_termos:
        while termo in texto_trabalho:
            idx = texto_trabalho.find(termo)
            prefixo = texto_trabalho[:idx].split()
            negado = any(w in negacoes for w in prefixo[-janela_negacao:])
            pol_efetiva = -polaridade if negado else polaridade
            if pol_efetiva > 0:
                n_pos += 1
            else:
                n_neg += 1
            texto_trabalho = texto_trabalho.replace(termo, ' ', 1)

    # Depois: palavras individuais
    palavras = texto_trabalho.split()
    set_pos = set(termos_pos)
    set_neg = set(termos_neg)

    for i, palavra in enumerate(palavras):
        if palavra in set_pos:
            polaridade = +1
        elif palavra in set_neg:
            polaridade = -1
        else:
            continue

        inicio = max(0, i - janela_negacao)
        contexto = palavras[inicio:i]
        negado = any(neg in contexto for neg in negacoes)
        pol_efetiva = -polaridade if negado else polaridade

        if pol_efetiva > 0:
            n_pos += 1
        else:
            n_neg += 1

    score = n_pos - n_neg
    total = n_pos + n_neg

    if score > 0:
        sentimento = 'POS'
    elif score < 0:
        sentimento = 'NEG'
    else:
        sentimento = 'NEU'

    confianca = round(abs(score) / total, 3) if total > 0 else 0.0
    return sentimento, score, n_pos, n_neg, confianca


# ── Teste rápido ─────────────────────────────────────────────────────────
testes = [
    ('adoro o cheiro desse produto, é maravilhoso', 'POS esperado'),
    ('detestei, ressecou minha pele e o cheiro é horrível', 'NEG esperado'),
    ('o produto tem textura cremosa e embalagem compacta', 'NEU esperado'),
    ('não gostei do produto, não recomendaria para ninguém', 'NEG esperado (negação)'),
    ('não é péssimo, mas poderia melhorar', 'NEU/POS esperado (negação de neg)'),
    ('não compraria, não voltaria e não recomendo', 'NEG esperado (múltiplas negações)'),
]
print('Teste da função léxica:\n')
for texto, esperado in testes:
    sent, score, np_, nn, conf = classificar_lexico(texto)
    print(f'  [{sent}] score={score:+d} (pos={np_}, neg={nn}, conf={conf:.2f}) | {esperado}')
    print(f'    "{texto}"')

Teste da função léxica:

  [POS] score=+2 (pos=2, neg=0, conf=1.00) | POS esperado
    "adoro o cheiro desse produto, é maravilhoso"
  [NEG] score=-3 (pos=0, neg=3, conf=1.00) | NEG esperado
    "detestei, ressecou minha pele e o cheiro é horrível"
  [NEU] score=+0 (pos=0, neg=0, conf=0.00) | NEU esperado
    "o produto tem textura cremosa e embalagem compacta"
  [NEG] score=-2 (pos=0, neg=2, conf=1.00) | NEG esperado (negação)
    "não gostei do produto, não recomendaria para ninguém"
  [POS] score=+1 (pos=1, neg=0, conf=1.00) | NEU/POS esperado (negação de neg)
    "não é péssimo, mas poderia melhorar"
  [NEG] score=-1 (pos=1, neg=2, conf=0.33) | NEG esperado (múltiplas negações)
    "não compraria, não voltaria e não recomendo"


In [7]:
# ── Aplicação do Léxico a Todos os Chunks ────────────────────────────────
LEXICO_CACHE = SENT_DIR / 'chunks_sentimento_lexico.parquet'

if LEXICO_CACHE.exists():
    df_lexico = pd.read_parquet(LEXICO_CACHE)
    print(f'Carregado do cache: {len(df_lexico):,} chunks (léxico)')
else:
    print(f'Classificando {len(df):,} chunks com abordagem léxica...')
    rows = []
    for _, row in df.iterrows():
        texto = str(row.get(TEXT_COL, ''))
        sent, score, n_pos, n_neg, conf = classificar_lexico(texto)
        rows.append({
            'chunk_id':          row['chunk_id'],
            'doc_id':            row.get('doc_id', ''),
            'projeto':           row.get('projeto', ''),
            'publico':           row.get('publico', ''),
            'sentimento_lexico': sent,
            'score_lexico':      score,
            'n_pos_lexico':      n_pos,
            'n_neg_lexico':      n_neg,
            'confianca_lexico':  conf,
        })
    df_lexico = pd.DataFrame(rows)
    df_lexico.to_parquet(LEXICO_CACHE, index=False)
    print(f'Classificação léxica concluída e salva em {LEXICO_CACHE}')

print(f'\nDistribuição geral (léxico):')
print(df_lexico['sentimento_lexico'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nScore léxico — estatísticas:')
print(df_lexico['score_lexico'].describe().round(2).to_string())

Carregado do cache: 4,976 chunks (léxico)

Distribuição geral (léxico):
sentimento_lexico
POS    81.4
NEU    11.9
NEG     6.7

Score léxico — estatísticas:
count    4976.00
mean        3.15
std         3.19
min       -16.00
25%         1.00
50%         3.00
75%         5.00
max        25.00


---
## 5. Sentimento por Projeto

Para cada projeto, calculamos a proporção de chunks positivos, negativos e neutros
pela **abordagem léxica** (abordagem principal do projeto, sem Deep Learning).

**Como interpretar:**
- Alto % NEG → entrevistados mais críticos: dor, reclamação, insatisfação
- Alto % POS → entrevistados mais entusiastas: prazer, recomendação, elogio
- Alto % NEU → discurso mais descritivo: hábitos, rotinas, processos

> Resultados baseados em **`sentimento_lexico`** e **`score_lexico`**.

In [8]:
# ── Proporção por projeto (sentimento léxico) ────────────────────────────
sent_proj = (
    df_lexico.groupby(['projeto', 'sentimento_lexico'])
    .size()
    .unstack(fill_value=0)
)
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_proj.columns:
        sent_proj[col] = 0

sent_proj_pct = sent_proj.div(sent_proj.sum(axis=1), axis=0).mul(100).round(1)
sent_proj_pct = sent_proj_pct[['POS', 'NEU', 'NEG']]
sent_proj_pct = sent_proj_pct.sort_values('NEG', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
cores = {'POS': '#2ecc71', 'NEU': '#95a5a6', 'NEG': '#e74c3c'}
bottom = np.zeros(len(sent_proj_pct))
for col in ['POS', 'NEU', 'NEG']:
    vals = sent_proj_pct[col].values
    ax.barh(sent_proj_pct.index, vals, left=bottom, color=cores[col], label=col)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 8:
            ax.text(b + v/2, i, f'{v:.0f}%', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax.set_xlabel('% de chunks')
ax.set_title('Distribuição de Sentimento por Projeto — Abordagem Léxica (Principal)', fontsize=13)
ax.legend(loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(SENT_DIR / 'sentimento_lexico_por_projeto.png', dpi=150, bbox_inches='tight')
plt.show()

sent_proj_pct.to_csv(SENT_DIR / 'sentimento_lexico_por_projeto.csv')
print('Tabela:')
display(sent_proj_pct)

Tabela:


sentimento_lexico,POS,NEU,NEG
projeto,,,
mercato_brasil,63.8,19.0,17.2
bem_maes,60.4,28.7,10.9
havana_iii,74.6,15.1,10.3
3cs_perfumes,74.0,18.1,7.8
natura_3cs,82.4,10.6,7.0
biome,84.3,10.2,5.5
radiosa,85.0,9.5,5.5
compras_digitais,82.5,12.8,4.7
rosacea,84.8,11.3,3.9


In [9]:
# ── Score léxico médio por projeto ───────────────────────────────────────
score_proj = (
    df_lexico.groupby('projeto')['score_lexico']
    .agg(['mean', 'median', 'std'])
    .round(2)
    .sort_values('mean')
)

fig, ax = plt.subplots(figsize=(10, 6))
cores_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in score_proj['mean']]
bars = ax.barh(score_proj.index, score_proj['mean'], color=cores_bar)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Score léxico médio  (n_termos_pos − n_termos_neg)')
ax.set_title('Score Léxico Médio por Projeto', fontsize=13)
for i, (idx, row) in enumerate(score_proj.iterrows()):
    offset = 0.05 if row['mean'] >= 0 else -0.05
    ha = 'left' if row['mean'] >= 0 else 'right'
    ax.text(row['mean'] + offset, i, f"{row['mean']:.2f}", va='center', ha=ha, fontsize=8)
plt.tight_layout()
plt.savefig(SENT_DIR / 'score_lexico_por_projeto.png', dpi=150, bbox_inches='tight')
plt.show()

score_proj.to_csv(SENT_DIR / 'score_lexico_por_projeto.csv')
print(score_proj.to_string())

                  mean  median   std
projeto                             
bem_maes          1.27     1.0  1.88
mercato_brasil    1.62     1.0  3.17
3cs_perfumes      2.03     2.0  2.22
havana_iii        2.20     2.0  2.48
compras_digitais  3.03     3.0  2.77
biome             3.16     3.0  2.87
rosacea           3.16     3.0  2.79
natura_3cs        3.31     3.0  3.32
gaia_ii           3.47     3.0  2.60
radiosa           3.68     3.0  3.40
anima             4.79     4.0  3.54
jack_pearson      5.54     5.0  4.01


---
## 6. Sentimento por Cluster

Cruzamos o sentimento com os 18 clusters do KMeans para entender quais
**tipos de discurso** estão associados a sentimentos positivos ou negativos.

Por exemplo:
- Cluster de 'reclamação sobre embalagem' → alto NEG
- Cluster de 'recomendação de produto' → alto POS
- Cluster de 'descrição de rotina de uso' → alto NEU

Isso é mais útil do que sentimento por projeto, porque revela
**sobre o que** as pessoas estão sendo positivas ou negativas.

In [10]:
# ── Junta sentimento léxico com cluster_label ─────────────────────────────
df_lexico_clusters = df_lexico.merge(
    df_clusters[['chunk_id', 'cluster_label', 'nmf_topic_id']],
    on='chunk_id', how='left'
)
print(f'Chunks com cluster atribuído: {df_lexico_clusters["cluster_label"].notna().sum():,}')

sent_cluster = (
    df_lexico_clusters.groupby(['cluster_label', 'sentimento_lexico'])
    .size()
    .unstack(fill_value=0)
)
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_cluster.columns:
        sent_cluster[col] = 0

sent_cluster_pct = sent_cluster.div(sent_cluster.sum(axis=1), axis=0).mul(100).round(1)
sent_cluster_pct = sent_cluster_pct[['POS', 'NEU', 'NEG']]
sent_cluster_pct['n_chunks'] = sent_cluster.sum(axis=1)
sent_cluster_pct = sent_cluster_pct.sort_values('NEG', ascending=False)

_label_path = CLUSTER_RUN / 'tables' / 'cluster_tfidf_labels.csv'
if _label_path.exists():
    df_labels = pd.read_csv(_label_path).set_index('cluster_label')
    label_map = df_labels['auto_label_short'].to_dict()
    sent_cluster_pct['label'] = sent_cluster_pct.index.map(
        lambda x: f"{x} | {label_map.get(x, '')[:30]}"
    )
else:
    sent_cluster_pct['label'] = sent_cluster_pct.index

fig, ax = plt.subplots(figsize=(8, max(8, len(sent_cluster_pct) * 0.45)))
heat_data = sent_cluster_pct[['POS', 'NEU', 'NEG']].values
im = ax.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['POS', 'NEU', 'NEG'], fontsize=11)
ax.set_yticks(range(len(sent_cluster_pct)))
ax.set_yticklabels(sent_cluster_pct['label'].values, fontsize=8)
for i in range(len(sent_cluster_pct)):
    for j, col in enumerate(['POS', 'NEU', 'NEG']):
        v = heat_data[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color='black')
plt.colorbar(im, ax=ax, label='% de chunks')
ax.set_title('Sentimento Léxico por Cluster (% de chunks)', fontsize=12)
plt.tight_layout()
plt.savefig(SENT_DIR / 'sentimento_lexico_por_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

sent_cluster_pct.to_csv(SENT_DIR / 'sentimento_lexico_por_cluster.csv')
print('Gráfico e tabela salvos.')

Chunks com cluster atribuído: 4,976
Gráfico e tabela salvos.


---
## 7. Sentimento por Tópico NMF

Cruzamos sentimento com os 18 tópicos NMF. A diferença entre cluster e tópico é:

- **Cluster** (KMeans): cada chunk pertence a exatamente 1 cluster — agrupamento duro
- **Tópico NMF** (nmf_topic_id): cada chunk tem um tópico dominante — mas pode ter
  ativação em múltiplos tópicos

Usar os dois dá uma visão mais completa de onde está o sentimento negativo no corpus.

In [11]:
# ── % de sentimento léxico por tópico NMF ───────────────────────────────
mask_nmf = df_lexico_clusters['nmf_topic_id'].notna()
sent_nmf = (
    df_lexico_clusters[mask_nmf]
    .groupby(['nmf_topic_id', 'sentimento_lexico'])
    .size()
    .unstack(fill_value=0)
)
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_nmf.columns:
        sent_nmf[col] = 0

sent_nmf_pct = sent_nmf.div(sent_nmf.sum(axis=1), axis=0).mul(100).round(1)
sent_nmf_pct = sent_nmf_pct[['POS', 'NEU', 'NEG']]
sent_nmf_pct['n_chunks'] = sent_nmf.sum(axis=1)

nmf_topics_df = pd.read_csv(CLUSTER_RUN / 'tables' / 'nmf_topics.csv')
nmf_label_map = dict(zip(nmf_topics_df['topic_id'],
                         nmf_topics_df.get('auto_label_short', nmf_topics_df['topic_id'])))
sent_nmf_pct['label'] = sent_nmf_pct.index.map(
    lambda x: f"{x} | {str(nmf_label_map.get(x, ''))[:30]}"
)
sent_nmf_pct = sent_nmf_pct.sort_values('NEG', ascending=False)

fig, ax = plt.subplots(figsize=(8, max(8, len(sent_nmf_pct) * 0.45)))
heat_data_nmf = sent_nmf_pct[['POS', 'NEU', 'NEG']].values
im2 = ax.imshow(heat_data_nmf, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['POS', 'NEU', 'NEG'], fontsize=11)
ax.set_yticks(range(len(sent_nmf_pct)))
ax.set_yticklabels(sent_nmf_pct['label'].values, fontsize=8)
for i in range(len(sent_nmf_pct)):
    for j in range(3):
        ax.text(j, i, f'{heat_data_nmf[i,j]:.0f}', ha='center', va='center', fontsize=8)
plt.colorbar(im2, ax=ax, label='% de chunks')
ax.set_title('Sentimento Léxico por Tópico NMF (% de chunks)', fontsize=12)
plt.tight_layout()
plt.savefig(SENT_DIR / 'sentimento_lexico_por_topico_nmf.png', dpi=150, bbox_inches='tight')
plt.show()

sent_nmf_pct.to_csv(SENT_DIR / 'sentimento_lexico_por_topico_nmf.csv')
print('Gráfico e tabela salvos.')

Gráfico e tabela salvos.


---
## 8. Exemplos Reais — Trechos Mais Positivos e Mais Negativos

Para cada projeto, extraímos os **3 chunks mais positivos** (maior prob_pos)
e os **3 chunks mais negativos** (maior prob_neg).

Isso serve como evidência citável no relatório final — mostra
o que especificamente as pessoas disseram de melhor e de pior sobre cada projeto.

In [12]:
# ── Top 3 mais positivos e mais negativos por projeto (léxico) ────────────
df_lexico_full = df_lexico.merge(df[[TEXT_COL, 'chunk_id']], on='chunk_id', how='left')

exemplos_rows = []
for proj in sorted(df_lexico_full['projeto'].dropna().unique()):
    sub = df_lexico_full[df_lexico_full['projeto'] == proj]

    for rank, (_, row) in enumerate(sub.nlargest(3, 'score_lexico').iterrows(), 1):
        exemplos_rows.append({
            'projeto': proj, 'tipo': 'mais_positivo', 'rank': rank,
            'sentimento_lexico': row['sentimento_lexico'],
            'score_lexico':      row['score_lexico'],
            'confianca_lexico':  row['confianca_lexico'],
            'chunk_id':          row['chunk_id'],
            'trecho':            str(row.get(TEXT_COL, ''))[:400],
        })

    for rank, (_, row) in enumerate(sub.nsmallest(3, 'score_lexico').iterrows(), 1):
        exemplos_rows.append({
            'projeto': proj, 'tipo': 'mais_negativo', 'rank': rank,
            'sentimento_lexico': row['sentimento_lexico'],
            'score_lexico':      row['score_lexico'],
            'confianca_lexico':  row['confianca_lexico'],
            'chunk_id':          row['chunk_id'],
            'trecho':            str(row.get(TEXT_COL, ''))[:400],
        })

df_exemplos = pd.DataFrame(exemplos_rows)
df_exemplos.to_csv(SENT_DIR / 'exemplos_sentimento_lexico_por_projeto.csv', index=False)
print(f'{len(df_exemplos)} exemplos salvos')

proj_ex = df_lexico_full['projeto'].dropna().value_counts().idxmax()
print(f'\nExemplos do projeto com mais chunks: {proj_ex}')
display(
    df_exemplos[df_exemplos['projeto'] == proj_ex]
    [['tipo', 'rank', 'sentimento_lexico', 'score_lexico', 'confianca_lexico', 'trecho']]
)

72 exemplos salvos

Exemplos do projeto com mais chunks: natura_3cs


,tipo,rank,sentimento_lexico,score_lexico,confianca_lexico,trecho
54,mais_positivo,1,POS,18,0.818,"Tem o dia dos namorados, tem o dia dos avós, dia dos pais, tudo vai presente aqui. Aqui não sei pra onde correr, né?..."
55,mais_positivo,2,POS,18,1.000,"Boa. E o presente de amanhã que a gente vai comprar, você já sabe onde é a loja? Já escolheu ou você vai lá e já sab..."
56,mais_positivo,3,POS,17,1.000,"Que legal, você faz curso do quê? Então, eu comecei a fazer curso de costura, né, para as bolsas. Eu comecei agora m..."
57,mais_negativo,1,NEG,-5,0.556,"Já aconteceu também, né, de você deixar pra última hora, mas eu costumo, pelo menos, me planejar uma semana, dez dia..."
58,mais_negativo,2,NEG,-4,1.000,"Então, assim, eu tenho bastante dificuldade de estar pedindo, igual eu pedi acho que 30, caixas, aí você acaba vendo..."
59,mais_negativo,3,NEG,-4,0.667,"Não desmonta o kit. Não desmonta. Veio aquele kit, veio aquele kit. Agora, se você quer comprar uma outra coisa para..."


---
## 9. Experimento Adicional com Modelo BERT/Transformer

> ⚠️ **Esta seção usa Deep Learning (modelo Transformer via `pysentimiento`).**
> Por isso, **não é a abordagem principal do projeto** — serve apenas como comparação exploratória.
>
> Para executar, defina `RODAR_BERT = True` na célula de configuração (seção 3).
> Com `RODAR_BERT = False` (padrão), esta seção é completamente ignorada.

### O que é pysentimiento?

`pysentimiento` encapsula o modelo `robertuito` (RoBERTa treinado em ~60M tweets PT/ES).
Retorna probabilidades para POS, NEG e NEU considerando o contexto completo da frase.

**Por que não usamos como abordagem principal?**
- Requer Deep Learning (modelo Transformer de ~500 MB)
- Demora ~3–4 horas em CPU para processar 4.976 chunks
- Menor interpretabilidade — não é possível saber quais palavras geraram a classificação

**Por que manter como experimento?**
- Pode capturar nuances contextuais que o léxico não alcança
- Permite validar se o léxico está classificando corretamente os casos mais claros
- A tabela de concordância léxico × BERT revela onde os métodos divergem

As colunas geradas são: `sentimento_bert`, `confianca_bert`, `prob_pos_bert`, `prob_neg_bert`, `prob_neu_bert`

In [13]:
# ── Experimento BERT — pysentimiento ─────────────────────────────────────
# Esta célula só executa se RODAR_BERT = True

if RODAR_BERT:
    try:
        from pysentimiento import create_analyzer
        print('pysentimiento já instalado.')
    except ImportError:
        import subprocess, sys
        print('Instalando pysentimiento...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pysentimiento', '-q'])
        from pysentimiento import create_analyzer
        print('Instalado com sucesso.')

    try:
        from tqdm.auto import tqdm
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])
        from tqdm.auto import tqdm

    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Dispositivo: {device.upper()}')

    BERT_CACHE = SENT_DIR / 'chunks_sentimento_bert.parquet'

    if BERT_CACHE.exists():
        df_bert = pd.read_parquet(BERT_CACHE)
        print(f'Carregado do cache BERT: {len(df_bert):,} chunks')
    else:
        print('Carregando modelo BERT (pode demorar ~2 min no primeiro download)...')
        analyzer = create_analyzer(task='sentiment', lang='pt')
        print(f'Modelo carregado. Classificando {len(df):,} chunks...')

        textos    = [str(row[TEXT_COL])[:512] for _, row in df.iterrows()]
        chunk_ids = df['chunk_id'].tolist()

        BATCH_SIZE = 32
        rows_bert  = []
        for start in tqdm(range(0, len(textos), BATCH_SIZE), desc='BERT inference'):
            batch_textos = textos[start:start + BATCH_SIZE]
            batch_ids    = chunk_ids[start:start + BATCH_SIZE]
            resultados   = analyzer.predict(batch_textos)
            for cid, r in zip(batch_ids, resultados):
                probas = {k.upper(): v for k, v in r.probas.items()}
                rows_bert.append({
                    'chunk_id':        cid,
                    'sentimento_bert': r.output.upper(),
                    'prob_pos_bert':   round(probas.get('POS', 0.0), 4),
                    'prob_neg_bert':   round(probas.get('NEG', 0.0), 4),
                    'prob_neu_bert':   round(probas.get('NEU', 0.0), 4),
                    'confianca_bert':  round(max(probas.values()), 4),
                })

        df_bert = pd.DataFrame(rows_bert)
        df_bert.to_parquet(BERT_CACHE, index=False)
        print(f'Classificação BERT concluída e salva em {BERT_CACHE}')

    print(f'\nDistribuição BERT (pysentimiento):')
    print(df_bert['sentimento_bert'].value_counts(normalize=True).mul(100).round(1).to_string())
else:
    print('RODAR_BERT = False — experimento BERT ignorado.')
    print('Para executar: defina RODAR_BERT = True na célula de configuração (seção 3) e re-execute.')


pysentimiento já instalado.
Dispositivo: CPU
Carregando modelo BERT (pode demorar ~2 min no primeiro download)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9996.03it/s]


Modelo carregado. Classificando 4,976 chunks...


BERT inference: 100%|██████████| 156/156 [09:35<00:00,  3.69s/it]

Classificação BERT concluída e salva em c:\Users\daiki\Desktop\kyrapesquisa-main\kyrapesquisa-main\Kyra Atual\outputs\sentimento\chunks_sentimento_bert.parquet

Distribuição BERT (pysentimiento):
sentimento_bert
NEU    78.8
POS    12.8
NEG     8.4


In [14]:
# ── Comparação Léxico × BERT ─────────────────────────────────────────────
# Só executa se RODAR_BERT = True e a classificação BERT estiver disponível

if RODAR_BERT and 'df_bert' in dir():
    df_comparacao = df_lexico[
        ['chunk_id', 'projeto', 'sentimento_lexico', 'score_lexico', 'confianca_lexico']
    ].merge(
        df_bert[['chunk_id', 'sentimento_bert', 'confianca_bert',
                 'prob_pos_bert', 'prob_neg_bert', 'prob_neu_bert']],
        on='chunk_id', how='left'
    )
    df_comparacao['concordancia'] = df_comparacao.apply(
        lambda r: 'concordam' if r['sentimento_lexico'] == r['sentimento_bert'] else 'divergem',
        axis=1
    )

    taxa_concordancia = (df_comparacao['concordancia'] == 'concordam').mean() * 100
    print(f'Taxa de concordância léxico × BERT: {taxa_concordancia:.1f}%\n')

    print('Concordância por classe (léxico):')
    for sent in ['POS', 'NEG', 'NEU']:
        sub = df_comparacao[df_comparacao['sentimento_lexico'] == sent]
        if len(sub) > 0:
            taxa = (sub['sentimento_bert'] == sent).mean() * 100
            print(f'  {sent}: {taxa:.1f}% concordam ({len(sub):,} chunks)')

    divergentes = df_comparacao[df_comparacao['concordancia'] == 'divergem']
    print(f'\nAmostra de {min(10, len(divergentes))} divergências ({len(divergentes):,} total):')
    display(
        divergentes[['projeto', 'sentimento_lexico', 'score_lexico',
                     'sentimento_bert', 'confianca_bert', 'concordancia']]
        .sample(min(10, len(divergentes)), random_state=42)
    )

    df_comparacao.to_csv(SENT_DIR / 'comparacao_lexico_bert.csv', index=False)
    print('\nTabela completa salva em comparacao_lexico_bert.csv')

    fig, ax = plt.subplots(figsize=(6, 5))
    crosstab = pd.crosstab(df_comparacao['sentimento_lexico'], df_comparacao['sentimento_bert'])
    sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('BERT (pysentimiento)')
    ax.set_ylabel('Léxico (abordagem principal)')
    ax.set_title(f'Léxico × BERT — {taxa_concordancia:.1f}% concordam')
    plt.tight_layout()
    plt.savefig(SENT_DIR / 'comparacao_lexico_bert.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Comparação léxico × BERT não disponível (RODAR_BERT = False).')
    print('Para ver a comparação: defina RODAR_BERT = True na célula de configuração (seção 3) e re-execute.')

Taxa de concordância léxico × BERT: 23.9%

Concordância por classe (léxico):
  POS: 14.7% concordam (4,048 chunks)
  NEG: 23.6% concordam (335 chunks)
  NEU: 86.8% concordam (593 chunks)

Amostra de 10 divergências (3,786 total):


,projeto,sentimento_lexico,score_lexico,sentimento_bert,confianca_bert,concordancia
2089,radiosa,POS,1,NEU,0.8210,divergem
2972,natura_3cs,POS,2,NEU,0.4803,divergem
337,rosacea,POS,2,NEU,0.8327,divergem
3838,mercato_brasil,POS,1,NEU,0.8565,divergem
2232,gaia_ii,POS,7,NEU,0.8279,divergem
4774,compras_digitais,POS,5,NEU,0.7696,divergem
835,3cs_perfumes,POS,4,NEU,0.7459,divergem
3702,natura_3cs,POS,3,NEG,0.7008,divergem
4367,compras_digitais,POS,6,NEU,0.7647,divergem
2206,gaia_ii,POS,5,NEU,0.6968,divergem



Tabela completa salva em comparacao_lexico_bert.csv


### 9b. Comparação por Projeto — Léxico × BERT

Verifica se o **ranking de projetos por negatividade** é consistente entre os dois métodos.
Se BERT confirmar a mesma ordem, a análise léxica fica validada.


In [15]:
# ── Comparação por Projeto — Léxico × BERT ───────────────────────────────
if RODAR_BERT and 'df_bert' in dir():

    df_comp_proj = df_lexico[['chunk_id', 'projeto', 'sentimento_lexico']].merge(
        df_bert[['chunk_id', 'sentimento_bert', 'prob_pos_bert', 'prob_neg_bert']],
        on='chunk_id', how='left'
    )

    pct_neg_lexico = (
        df_comp_proj.groupby('projeto')['sentimento_lexico']
        .apply(lambda x: (x == 'NEG').mean() * 100).rename('pct_neg_lexico')
    )
    pct_neg_bert = (
        df_comp_proj.groupby('projeto')['sentimento_bert']
        .apply(lambda x: (x == 'NEG').mean() * 100).rename('pct_neg_bert')
    )
    pct_pos_lexico = (
        df_comp_proj.groupby('projeto')['sentimento_lexico']
        .apply(lambda x: (x == 'POS').mean() * 100).rename('pct_pos_lexico')
    )
    pct_pos_bert = (
        df_comp_proj.groupby('projeto')['sentimento_bert']
        .apply(lambda x: (x == 'POS').mean() * 100).rename('pct_pos_bert')
    )

    df_proj_comp = pd.concat([pct_neg_lexico, pct_neg_bert, pct_pos_lexico, pct_pos_bert], axis=1).round(1)
    df_proj_comp['rank_neg_lexico'] = df_proj_comp['pct_neg_lexico'].rank(ascending=False).astype(int)
    df_proj_comp['rank_neg_bert']   = df_proj_comp['pct_neg_bert'].rank(ascending=False).astype(int)
    df_proj_comp['rank_diff']       = (df_proj_comp['rank_neg_lexico'] - df_proj_comp['rank_neg_bert']).abs()
    df_proj_comp = df_proj_comp.sort_values('pct_neg_lexico', ascending=False)

    print('Comparação % NEG por projeto (léxico vs BERT):')
    display(df_proj_comp)

    from scipy.stats import spearmanr
    corr, pval = spearmanr(df_proj_comp['pct_neg_lexico'], df_proj_comp['pct_neg_bert'])
    print(f'\nCorrelação de Spearman (ranking NEG%): r={corr:.3f}, p={pval:.4f}')
    if corr > 0.8:
        print('  -> Alta concordância: BERT confirma o ranking do léxico')
    elif corr > 0.5:
        print('  -> Concordância moderada: rankings parecidos com algumas divergências')
    else:
        print('  -> Baixa concordância: os métodos discordam na ordem dos projetos')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    x = np.arange(len(df_proj_comp))
    w = 0.35

    ax = axes[0]
    ax.barh(x + w/2, df_proj_comp['pct_neg_lexico'], w, label='Léxico', color='#3498db', alpha=0.85)
    ax.barh(x - w/2, df_proj_comp['pct_neg_bert'],   w, label='BERT',   color='#e67e22', alpha=0.85)
    ax.set_yticks(x)
    ax.set_yticklabels(df_proj_comp.index, fontsize=9)
    ax.set_xlabel('% chunks NEG')
    ax.set_title('% Negativo por Projeto — Léxico vs BERT', fontsize=11)
    ax.legend()

    df_pos = df_proj_comp.sort_values('pct_pos_lexico')
    x2 = np.arange(len(df_pos))
    ax2 = axes[1]
    ax2.barh(x2 + w/2, df_pos['pct_pos_lexico'], w, label='Léxico', color='#2ecc71', alpha=0.85)
    ax2.barh(x2 - w/2, df_pos['pct_pos_bert'],   w, label='BERT',   color='#f39c12', alpha=0.85)
    ax2.set_yticks(x2)
    ax2.set_yticklabels(df_pos.index, fontsize=9)
    ax2.set_xlabel('% chunks POS')
    ax2.set_title('% Positivo por Projeto — Léxico vs BERT', fontsize=11)
    ax2.legend()

    plt.suptitle(f'Comparação Léxico × BERT por Projeto  |  Spearman r={corr:.2f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(SENT_DIR / 'comparacao_lexico_bert_por_projeto.png', dpi=150, bbox_inches='tight')
    plt.show()

    df_proj_comp.to_csv(SENT_DIR / 'comparacao_lexico_bert_por_projeto.csv')
    print('Gráfico e tabela salvos.')
else:
    print('Comparação por projeto não disponível (RODAR_BERT = False).')


Comparação % NEG por projeto (léxico vs BERT):


,pct_neg_lexico,pct_neg_bert,pct_pos_lexico,pct_pos_bert,rank_neg_lexico,rank_neg_bert,rank_diff
projeto,,,,,,,
mercato_brasil,17.2,16.5,63.8,5.8,1,1,0
bem_maes,10.9,7.9,60.4,19.8,2,5,3
havana_iii,10.3,6.3,74.6,6.3,3,8,5
3cs_perfumes,7.8,6.4,74.0,6.8,4,7,3
natura_3cs,7.0,7.7,82.4,13.0,5,6,1
biome,5.5,10.6,84.3,8.1,6,3,3
radiosa,5.5,10.1,85.0,13.1,6,4,2
compras_digitais,4.7,5.1,82.5,6.5,8,10,2
rosacea,3.9,3.7,84.8,21.7,9,12,3



Correlação de Spearman (ranking NEG%): r=0.494, p=0.1027
  -> Baixa concordância: os métodos discordam na ordem dos projetos
Gráfico e tabela salvos.


---
## 10. Export Final

Consolida todos os outputs em arquivos prontos para o notebook 07 (resumo estruturado por template).

**Colunas principais (léxico — abordagem oficial):** `sentimento_lexico`, `score_lexico`, `confianca_lexico`
**Colunas BERT (quando `RODAR_BERT = True`):** `sentimento_bert`, `confianca_bert`, `prob_pos_bert`, etc.

**Arquivos gerados em `outputs/sentimento/`:**

| Arquivo | Conteúdo |
|---|---|
| `chunks_sentimento_lexico.parquet` | Todos os chunks com sentimento léxico (principal) |
| `sentimento_lexico_por_projeto.csv` | % POS/NEU/NEG por projeto (léxico) |
| `score_lexico_por_projeto.csv` | Score léxico médio por projeto |
| `sentimento_lexico_por_cluster.csv` | % POS/NEU/NEG por cluster KMeans (léxico) |
| `sentimento_lexico_por_topico_nmf.csv` | % POS/NEU/NEG por tópico NMF (léxico) |
| `exemplos_sentimento_lexico_por_projeto.csv` | Top 3 positivos + negativos por projeto |
| `sentimento_resumo_geral.json` | Resumo consolidado para o notebook 07 |
| `chunks_sentimento_bert.parquet` | Chunks com sentimento BERT (apenas se `RODAR_BERT = True`) |
| `comparacao_lexico_bert.csv` | Comparação léxico × BERT (apenas se `RODAR_BERT = True`) |

In [16]:
# ── Resumo consolidado em JSON (input do notebook 07) ─────────────────────
resumo_sentimento = {}

for proj in sorted(df_lexico['projeto'].dropna().unique()):
    sub = df_lexico[df_lexico['projeto'] == proj]
    vc  = sub['sentimento_lexico'].value_counts()
    total = len(sub)
    resumo_sentimento[proj] = {
        'n_chunks':              total,
        'pct_pos':               round(vc.get('POS', 0) / total * 100, 1),
        'pct_neg':               round(vc.get('NEG', 0) / total * 100, 1),
        'pct_neu':               round(vc.get('NEU', 0) / total * 100, 1),
        'score_lexico_medio':    round(sub['score_lexico'].mean(), 2),
        'score_lexico_mediana':  round(sub['score_lexico'].median(), 2),
        'sentimento_dominante':  vc.idxmax() if len(vc) > 0 else 'NEU',
        'metodo':                'lexico',
    }

with open(SENT_DIR / 'sentimento_resumo_geral.json', 'w', encoding='utf-8') as f:
    json.dump(resumo_sentimento, f, ensure_ascii=False, indent=2)

print('Export concluído. Arquivos em outputs/sentimento/:')
for f in sorted(SENT_DIR.glob('*')):
    print(f'  {f.name}')

print('\nResumo por projeto (sentimento léxico):')
df_resumo_final = pd.DataFrame(resumo_sentimento).T
display(df_resumo_final.sort_values('pct_neg', ascending=False))

Export concluído. Arquivos em outputs/sentimento/:
  chunks_sentimento_bert.parquet
  chunks_sentimento_lexico.parquet
  comparacao_lexico_bert.csv
  comparacao_lexico_bert.png
  comparacao_lexico_bert_por_projeto.csv
  comparacao_lexico_bert_por_projeto.png
  exemplos_sentimento_lexico_por_projeto.csv
  score_lexico_por_projeto.csv
  score_lexico_por_projeto.png
  sentimento_lexico_por_cluster.csv
  sentimento_lexico_por_cluster.png
  sentimento_lexico_por_projeto.csv
  sentimento_lexico_por_projeto.png
  sentimento_lexico_por_topico_nmf.csv
  sentimento_lexico_por_topico_nmf.png
  sentimento_resumo_geral.json

Resumo por projeto (sentimento léxico):


,n_chunks,pct_pos,pct_neg,pct_neu,score_lexico_medio,score_lexico_mediana,sentimento_dominante,metodo
mercato_brasil,516,63.8,17.2,19.0,1.62,1.0,POS,lexico
bem_maes,101,60.4,10.9,28.7,1.27,1.0,POS,lexico
havana_iii,126,74.6,10.3,15.1,2.2,2.0,POS,lexico
3cs_perfumes,485,74.0,7.8,18.1,2.03,2.0,POS,lexico
natura_3cs,985,82.4,7.0,10.6,3.31,3.0,POS,lexico
biome,236,84.3,5.5,10.2,3.16,3.0,POS,lexico
radiosa,474,85.0,5.5,9.5,3.68,3.0,POS,lexico
compras_digitais,492,82.5,4.7,12.8,3.03,3.0,POS,lexico
rosacea,539,84.8,3.9,11.3,3.16,3.0,POS,lexico
anima,406,91.6,3.7,4.7,4.79,4.0,POS,lexico


---
## 11. Resumo dos Resultados

In [17]:
print('=' * 65)
print('RESUMO — NOTEBOOK 06: ANÁLISE DE SENTIMENTO')
print('=' * 65)
print(f'\n[Abordagem principal: LÉXICA (sem Deep Learning)]')
print(f'[Experimento BERT: {"EXECUTADO" if RODAR_BERT else "NÃO EXECUTADO (RODAR_BERT = False)"}]')

if 'df_lexico' in dir():
    dist = df_lexico['sentimento_lexico'].value_counts(normalize=True).mul(100).round(1)
    print(f'\n[Distribuição Geral — {len(df_lexico):,} chunks — Léxico]')
    for label, pct in dist.items():
        print(f'  {label}: {pct:.1f}%')
    print(f'\n  Score léxico médio geral : {df_lexico["score_lexico"].mean():.2f}')
    print(f'  Score léxico mediana     : {df_lexico["score_lexico"].median():.2f}')

if 'resumo_sentimento' in dir():
    mais_pos = max(resumo_sentimento, key=lambda p: resumo_sentimento[p]['pct_pos'])
    mais_neg = max(resumo_sentimento, key=lambda p: resumo_sentimento[p]['pct_neg'])
    print(f'\n[Projetos — Léxico]')
    print(f'  Mais positivo : {mais_pos} ({resumo_sentimento[mais_pos]["pct_pos"]:.1f}% POS, '
          f'score médio = {resumo_sentimento[mais_pos]["score_lexico_medio"]:+.2f})')
    print(f'  Mais negativo : {mais_neg} ({resumo_sentimento[mais_neg]["pct_neg"]:.1f}% NEG, '
          f'score médio = {resumo_sentimento[mais_neg]["score_lexico_medio"]:+.2f})')

if RODAR_BERT and 'df_comparacao' in dir():
    taxa = (df_comparacao['concordancia'] == 'concordam').mean() * 100
    print(f'\n[Comparação Léxico × BERT]')
    print(f'  Taxa de concordância: {taxa:.1f}%')

print(f'\n[Outputs salvos em outputs/sentimento/]')
for f in sorted(SENT_DIR.glob('*')):
    print(f'  {f.name}')

print('\n→ Próximo passo: notebook 07 — Resumo estruturado por template')

RESUMO — NOTEBOOK 06: ANÁLISE DE SENTIMENTO

[Abordagem principal: LÉXICA (sem Deep Learning)]
[Experimento BERT: EXECUTADO]

[Distribuição Geral — 4,976 chunks — Léxico]
  POS: 81.4%
  NEU: 11.9%
  NEG: 6.7%

  Score léxico médio geral : 3.15
  Score léxico mediana     : 3.00

[Projetos — Léxico]
  Mais positivo : jack_pearson (95.5% POS, score médio = +5.54)
  Mais negativo : mercato_brasil (17.2% NEG, score médio = +1.62)

[Comparação Léxico × BERT]
  Taxa de concordância: 23.9%

[Outputs salvos em outputs/sentimento/]
  chunks_sentimento_bert.parquet
  chunks_sentimento_lexico.parquet
  comparacao_lexico_bert.csv
  comparacao_lexico_bert.png
  comparacao_lexico_bert_por_projeto.csv
  comparacao_lexico_bert_por_projeto.png
  exemplos_sentimento_lexico_por_projeto.csv
  score_lexico_por_projeto.csv
  score_lexico_por_projeto.png
  sentimento_lexico_por_cluster.csv
  sentimento_lexico_por_cluster.png
  sentimento_lexico_por_projeto.csv
  sentimento_lexico_por_projeto.png
  sentimento_